# V2G ABM smoke test  (interactive)

**Purpose**: prove the V2G agent-based model deploys and runs correctly on Oxford ARC.  This is the *interactive notes* deliverable requested by David at Meeting 6.

**What this notebook does**: walks through the model in five small steps.  Each step is one cell.  Run the cells in order and read the markdown between them.

1. Confirm Python + repository are available
2. Build one EVAgent of each typology and inspect its state
3. Step the agent through a single day and print its hour-by-hour decisions
4. Run a small two-country fleet sweep (Israel + UK, V0/V1G/V2G) and report fleet totals
5. Display the literature-anchored battery aging table

Run time on a single ARC CPU: under one minute total.

## 1.  Environment check

In [ ]:
import sys, os, platform
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('Repo root', os.path.abspath('..'))
for module in ('numpy', 'matplotlib', 'pandas'):
    try:
        __import__(module)
        print(f'  {module}: OK')
    except ImportError:
        print(f'  {module}: MISSING')

## 2.  Build one EVAgent of each typology

In [ ]:
import sys
sys.path.insert(0, '..')
import src.agents.ev_agent as ev
from src.agents.ev_agent import (
    EVAgent, ALL_TYPOLOGIES,
    DAILY_CHARGER, PUBLIC_CHARGER, BEV_2ND_VEHICLE, THRESHOLD_CHARGER,
    COUNTERFACTUAL_V2G,
)
ev.SEM_ENABLED = True
agents = []
for typ in ALL_TYPOLOGIES:
    a = EVAgent(agent_id=hash(typ) % 10_000,
                typology=typ,
                counterfactual=COUNTERFACTUAL_V2G)
    agents.append(a)
    print(f'{typ:>20} | home_charger={a.state.has_home_charger} | '
          f'v2g_capable={a.state.v2g_capable} | '
          f'opted_in={a.state.v2g_opted_in} | '
          f'OSP={a.state.osp:.2f}')

## 3.  Step one agent through 24 hours

Pick the Daily Charger, run it through Sunday hour-by-hour, print every charge / discharge decision.

In [ ]:
from src.pricing import price_at_hour

daily = agents[0]
print(f'Agent {daily.id} ({daily.typology}) - hour-by-hour for Sunday in July')
print(f'{"hour":>4} | {"price":>6} | {"status":>16} | {"kWh":>6} | {"NIS":>7} | {"SoC":>5}')
print('-' * 55)
for hour in range(24):
    p = price_at_hour(hour, 0, 7)
    daily.step(current_hour=hour, current_price_per_kwh=p,
               discharge_revenue_per_kwh=p, month=7)
    last = daily.hourly_log[-1]
    print(f'{hour:>4} | {p:>6.3f} | {last["status"]:>16} | '
          f'{last["energy_kwh"]:>+6.2f} | {-last["cost_currency"]:>+7.2f} | '
          f'{daily.state.soc:>5.2f}')

## 4.  Two-country fleet sweep

Run 80 agents through a full simulated year for Israel + UK x V0/V1G/V2G (six combinations, about 15 seconds total on a single CPU).

In [ ]:
from src.smoke_w10d_twocountry import summarise, COUNTERFACTUALS
import time

t0 = time.time()
rows = []
for country in ('Israel', 'UK'):
    for cf in COUNTERFACTUALS:
        rows.append(summarise(country, cf))
print(f'Total wall time: {time.time()-t0:.1f}s')
print()
print(f'{"Country":>8} | {"CF":>4} | {"V2G EVs":>8} | {"V2G kWh/yr":>11} | {"Net":>12}')
print('-' * 60)
for r in rows:
    ccy = 'NIS' if r['country'] == 'Israel' else 'GBP'
    print(f'{r["country"]:>8} | {r["counterfactual"]:>4} | '
          f'{r["n_v2g_opted"]:>8,} | {r["v2g_kwh_yr"]:>11,.0f} | '
          f'{r["net_yr"]:>+8,.0f} {ccy}')

## 5.  Literature-anchored battery aging table

Aging numbers come from Wong et al. 2026 published values, scaled by observed-to-Wong V2G volume.  No simulation.

In [ ]:
import subprocess
out = subprocess.run([sys.executable, '-m', 'src.aging_table_lit'],
                     capture_output=True, text=True, cwd='..')
print(out.stdout)

## Done

If all five cells ran without error, the model deploys correctly on this machine.  Save this notebook with outputs to confirm the smoke test for the supervisor.